# Adaptive Inference Router - Google Colab Demo

This notebook demonstrates the **Adaptive Inference Router** based on the Intention Collapse framework.

## What does the router do?
- Measures **intention entropy H_int(I)** before generating responses
- **Low entropy** → direct answer (cheap, fast)
- **High entropy** → Chain-of-Thought (more tokens, better reasoning)

## 🚀 Usage:
1. **Run the installation cell** (next cell) - fully automatic
2. **Load the model** (GPT-2 for quick demo)
3. **Run 4 test questions** and observe router decisions
4. **Visualize results** with entropy and token usage graphs

## ⏱️ Total time: ~3-5 minutes

**Let's begin!** 👇

## 1. Complete Setup: Installation and Verification

**This cell executes everything needed:**
- Clones the repository from GitHub
- Installs the package and dependencies
- Configures Python path
- Verifies everything works

**⏱️ Estimated time: ~30 seconds**

Simply run the next cell and wait for: **"🎉 INSTALLATION COMPLETE"**

In [ ]:
import sys
import subprocess

print("=" * 70)
print("AUTOMATIC INSTALLATION - Adaptive Inference Router")
print("=" * 70)

# 1. Clean previous installations
print("\n[1/4] Cleaning previous installation...")
subprocess.run(["rm", "-rf", "/content/intention-collapse-experiments"], 
               capture_output=True, check=False)

# 2. Clone repository
print("[2/4] Cloning repository from GitHub...")
result = subprocess.run(
    ["git", "clone", "-q", "-b", "v2-router-experiments",
     "https://github.com/patriciomvera/intention-collapse-experiments.git",
     "/content/intention-collapse-experiments"],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(f"❌ Clone error: {result.stderr}")
    raise RuntimeError("Git clone failed")

print("   ✅ Repository cloned")

# 3. Install package
print("[3/4] Installing intention-collapse package...")
result = subprocess.run(
    ["pip", "install", "-q", "-e", "/content/intention-collapse-experiments/"],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(f"⚠️  Pip install had warnings (but may still work):")
    if result.stderr:
        # Show only last lines if error is too long
        error_lines = result.stderr.split('\n')
        print('\n'.join(error_lines[-10:]))
else:
    print("   ✅ Package installed")

# 4. Configure Python path (critical for Colab)
print("[4/4] Configuring Python path...")
if '/content/intention-collapse-experiments' not in sys.path:
    sys.path.insert(0, '/content/intention-collapse-experiments')
    print("   ✅ Path configured")

# 5. Verify installation
print("\n" + "=" * 70)
print("INSTALLATION VERIFICATION")
print("=" * 70)

try:
    from src.router import AdaptiveInferenceRouter, RouteDecision
    from src.metrics import compute_intention_entropy
    from src.controls import self_consistency_baseline
    
    print("✅ All imports successful!")
    print(f"✅ RouteDecision available: {[r.value for r in RouteDecision]}")
    print(f"✅ AdaptiveInferenceRouter: {AdaptiveInferenceRouter.__name__}")
    
    print("\n" + "=" * 70)
    print("🎉 INSTALLATION COMPLETE - Ready to run!")
    print("=" * 70)
    print("\n➡️  Continue executing the following notebook cells\n")
    
except ImportError as e:
    print(f"\n❌ Import error: {e}")
    print("\n🔧 Debug information:")
    print(f"   sys.path[0]: {sys.path[0]}")
    import os
    if os.path.exists('/content/intention-collapse-experiments/src'):
        print(f"   src/ exists with: {os.listdir('/content/intention-collapse-experiments/src')[:5]}")
    raise

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load small model for demo
# Options: "gpt2", "facebook/opt-350m", "EleutherAI/pythia-410m"
MODEL_NAME = "gpt2"

print(f"Loading model: {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print(f"[OK] Model loaded on: {model.device}")
print(f"[OK] Using {'GPU' if torch.cuda.is_available() else 'CPU'}")

## 2. Load Model

We'll use a small model for the demo (you can change to larger models if you have GPU)

## 3. Initialize Router

The router uses intention entropy to automatically decide the inference strategy.

In [ ]:
# Create router with entropy thresholds
router = AdaptiveInferenceRouter(
    model=model,
    tokenizer=tokenizer,
    entropy_threshold_low=0.5,   # H_int < 0.5 → DIRECT
    entropy_threshold_high=1.2,  # H_int > 1.2 → CoT
    verbose=True
)

print("[OK] Router initialized")
print(f"    Entropy thresholds:")
print(f"      - Low:  {router.entropy_threshold_low} (below → DIRECT answer)")
print(f"      - High: {router.entropy_threshold_high} (above → CoT reasoning)")
print(f"    Decision logic:")
print(f"      - H_int < {router.entropy_threshold_low} → DIRECT")
print(f"      - {router.entropy_threshold_low} ≤ H_int < {router.entropy_threshold_high} → CoT (uncertain)")  
print(f"      - H_int ≥ {router.entropy_threshold_high} → CoT (very uncertain)")

## 4. Experiment: 4 Test Questions

We test the router with questions of different complexity levels.

In [ ]:
# Test questions with different difficulty levels
test_questions = [
    {
        "question": "What is 2 + 2?",
        "expected_route": "DIRECT",
        "reason": "Simple arithmetic - low entropy expected"
    },
    {
        "question": "If a train leaves Chicago at 3pm going 60mph and another leaves New York at 4pm going 80mph, when do they meet?",
        "expected_route": "COT",
        "reason": "Complex word problem - high entropy expected"
    },
    {
        "question": "What is the capital of France?",
        "expected_route": "DIRECT",
        "reason": "Factual knowledge - low entropy expected"
    },
    {
        "question": "A baker has 12 cookies. He gives 1/3 to his friend and eats 2. How many remain?",
        "expected_route": "COT",
        "reason": "Multi-step reasoning - high entropy expected"
    },
]

print(f"Testing router with {len(test_questions)} questions...\n")
print("="*80)

## 5. Run Router

Now we run the router on each question and observe its decisions.

In [ ]:
results = []

for i, test_case in enumerate(test_questions, 1):
    print(f"\n[Question {i}/{len(test_questions)}]")
    print(f"Q: {test_case['question']}")
    print(f"Expected route: {test_case['expected_route']} ({test_case['reason']})")
    print("-" * 80)
    
    # Run router
    result = router.generate(
        question=test_case['question'],
        max_tokens_cot=100  # Use max_tokens_cot instead of max_new_tokens
    )
    
    results.append({
        'question': test_case['question'],
        'expected_route': test_case['expected_route'],
        'actual_route': result.route_taken.value.upper(),
        'entropy': result.intention_entropy,
        'answer': result.extracted_answer,
        'tokens': result.output_tokens,
        'cost_estimate': result.total_tokens * 0.000001  # Rough estimate
    })
    
    # Show result
    print(f"\n[RESULT]")
    print(f"  Route taken: {result.route_taken.value.upper()}")
    print(f"  Intention entropy: {result.intention_entropy:.3f}")
    print(f"  Answer: {result.extracted_answer}")
    print(f"  Tokens: {result.output_tokens}")
    print(f"  Total tokens: {result.total_tokens}")
    
    match = "✓" if result.route_taken.value.upper() == test_case['expected_route'] else "✗"
    print(f"  Match expected: {match}")
    print("="*80)

print("\n[OK] All experiments completed!")

## 6. Results Summary

Quantitative analysis of router behavior.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Entropy por pregunta
colors = ['green' if r == 'DIRECT' else 'orange' for r in df['actual_route']]
ax1.bar(range(len(df)), df['entropy'], color=colors, alpha=0.7)
ax1.axhline(y=router.entropy_threshold, color='red', linestyle='--', 
            label=f'Threshold ({router.entropy_threshold})')
ax1.set_xlabel('Question Index')
ax1.set_ylabel('Intention Entropy H_int(I)')
ax1.set_title('Intention Entropy per Question')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Token usage por ruta
route_tokens = df.groupby('actual_route')['tokens'].mean()
ax2.bar(route_tokens.index, route_tokens.values, 
        color=['green', 'orange'], alpha=0.7)
ax2.set_xlabel('Route')
ax2.set_ylabel('Average Tokens')
ax2.set_title('Token Usage by Route')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("[OK] Visualization complete!")

import pandas as pd

# Create DataFrame with results
df = pd.DataFrame(results)

print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print("\nRouting Accuracy:")
correct = sum(df['expected_route'] == df['actual_route'])
total = len(df)
print(f"  {correct}/{total} correct ({100*correct/total:.1f}%)")

print("\nRoute Distribution:")
print(df['actual_route'].value_counts())

print("\nEntropy Statistics:")
print(f"  Mean: {df['entropy'].mean():.3f}")
print(f"  Std:  {df['entropy'].std():.3f}")
print(f"  Min:  {df['entropy'].min():.3f}")
print(f"  Max:  {df['entropy'].max():.3f}")

print("\nCost Analysis:")
total_cost = df['cost_estimate'].sum()
print(f"  Total cost: ${total_cost:.6f}")
print(f"  Avg per query: ${total_cost/len(df):.6f}")

print("\nDetailed Results:")
display(df[['question', 'actual_route', 'entropy', 'tokens', 'cost_estimate']])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Entropy per question
colors = ['green' if r == 'DIRECT' else 'orange' for r in df['actual_route']]
ax1.bar(range(len(df)), df['entropy'], color=colors, alpha=0.7)
ax1.axhline(y=router.entropy_threshold_low, color='green', linestyle='--', 
            label=f'Low threshold ({router.entropy_threshold_low})')
ax1.axhline(y=router.entropy_threshold_high, color='red', linestyle='--', 
            label=f'High threshold ({router.entropy_threshold_high})')
ax1.set_xlabel('Question Index')
ax1.set_ylabel('Intention Entropy H_int(I)')
ax1.set_title('Intention Entropy per Question')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Token usage by route
route_tokens = df.groupby('actual_route')['tokens'].mean()
ax2.bar(route_tokens.index, route_tokens.values, 
        color=['green', 'orange'], alpha=0.7)
ax2.set_xlabel('Route')
ax2.set_ylabel('Average Tokens')
ax2.set_title('Token Usage by Route')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("[OK] Visualization complete!")

# Test with your own question
custom_question = "What is 15 multiplied by 23?"  # Change this

print(f"Testing custom question: {custom_question}\n")

result = router.generate(
    question=custom_question,
    max_tokens_cot=150
)

print(f"Route: {result.route_taken.value.upper()}")
print(f"Entropy: {result.intention_entropy:.3f}")
print(f"Answer: {result.extracted_answer}")
print(f"Full response: {result.generated_text}")

In [ ]:
# Test with your own question
custom_question = "What is 15 multiplied by 23?"  # Change this

print(f"Testing custom question: {custom_question}\n")

result = router.route_and_generate(
    question=custom_question,
    max_new_tokens=150
)

print(f"Route: {result.route_taken.value.upper()}")
print(f"Entropy: {result.intention_entropy:.3f}")
print(f"Answer: {result.extracted_answer}")
print(f"Full response: {result.generated_text}")

import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Entropy per question
colors = ['green' if r == 'DIRECT' else 'orange' for r in df['actual_route']]
ax1.bar(range(len(df)), df['entropy'], color=colors, alpha=0.7)
ax1.axhline(y=router.entropy_threshold, color='red', linestyle='--', 
            label=f'Threshold ({router.entropy_threshold})')
ax1.set_xlabel('Question Index')
ax1.set_ylabel('Intention Entropy H_int(I)')
ax1.set_title('Intention Entropy per Question')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Token usage by route
route_tokens = df.groupby('actual_route')['tokens'].mean()
ax2.bar(route_tokens.index, route_tokens.values, 
        color=['green', 'orange'], alpha=0.7)
ax2.set_xlabel('Route')
ax2.set_ylabel('Average Tokens')
ax2.set_title('Token Usage by Route')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("[OK] Visualization complete!")